MIT License

Copyright (c) 2026 THALES

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# A* Solver Inference Timing Benchmark

Systematic measurement of A* solver inference time across resolutions.
These timings serve as the **exact solver reference** for comparing neural surrogate speedup ratios
in the PASSAGE NeurIPS paper.

**Resolutions:** 64x64, 256x256
<!-- TODO:RESOLUTION_UPDATE: add 128x128, 512x512, 1024x1024, 2048x2048, 4096x4096 -->

**Solver backends:**
- `grid:astar` (Numba JIT, our fastest solver — the one used for ground truth generation)
- `grid:dijkstra` (Numba JIT, for comparison)
- `networkx:astar` (pure Python, for baseline comparison)

**Metrics:**
- Mean solve time (ms)
- Std deviation
- Median
- P95 and P99 latencies
- Peak memory

In [ ]:
import json
import logging
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Suppress verbose logging
logging.basicConfig(level=logging.WARNING)

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from passage.generate import find_path  # noqa: E402
from passage.pathfinding_utils import NUMBA_AVAILABLE  # noqa: E402

print(f"Numba available: {NUMBA_AVAILABLE}")

# Configuration
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibration.json"
EXPORT_DIR = ROOT / "outputs" / "export"

# Load calibration
with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)
GLOBAL_MIN = float(calibration["global_min"])
GLOBAL_MAX = float(calibration["global_max"])

# Load config for cost model weights
with open(ROOT / "config" / "config.yaml") as f:
    config = yaml.safe_load(f)

In [ ]:
# Benchmark parameters
RESOLUTIONS = [64, 256]  # TODO:RESOLUTION_UPDATE: add 128, 512, 1024, 2048, 4096
N_SAMPLES = 100  # Number of samples per resolution
COST_MODEL = "energy"  # The default cost model used for labels in passage-vision

# Solver configurations to benchmark
SOLVERS = [
    {
        "name": "grid:astar",
        "backend": "grid",
        "solver_name": "astar",
        "diagonal_movement": "always",
        "heuristic": "euclidean",
        "cost_model": COST_MODEL,
        "weight": config.get("solve", {}).get("cost_weight", {}).get(COST_MODEL, 1.0),
    },
    {
        "name": "grid:dijkstra",
        "backend": "grid",
        "solver_name": "dijkstra",
        "diagonal_movement": "always",
        "heuristic": "euclidean",
        "cost_model": COST_MODEL,
        "weight": config.get("solve", {}).get("cost_weight", {}).get(COST_MODEL, 1.0),
    },
]

# Only include networkx for small resolutions (too slow for 512+)
# for res in RESOLUTIONS:
#     if res <= 256:
#         SOLVERS.append({
#             "name": f"networkx:astar",
#             "backend": "networkx",
#             ...
#         })

print(f"Benchmarking {len(SOLVERS)} solvers at resolutions: {RESOLUTIONS}")
print(f"Samples per resolution: {N_SAMPLES}")
print(f"Cost model: {COST_MODEL}")

## 1. Generate Test Samples

We load samples from the exported parquet data to ensure we benchmark on representative terrain.

In [ ]:
import pyarrow.parquet as pq
import glob as globlib
import zstandard as zstd
import io
import torch


def decode_tensor_blob(blob: bytes) -> np.ndarray:
    """Decode a zstandard-compressed tensor blob from parquet."""
    dctx = zstd.ZstdDecompressor()
    decompressed = dctx.decompress(blob)
    buf = io.BytesIO(decompressed)
    tensor = torch.load(buf, weights_only=True, map_location="cpu")
    return tensor.numpy()


def load_samples(resolution: int, split: str = "test", n: int = 100) -> list[dict]:
    """Load samples from parquet export."""
    res_dir = EXPORT_DIR / f"{resolution}x{resolution}"
    files = sorted(globlib.glob(str(res_dir / split / "*.parquet")))

    if not files:
        # Fall back to train split
        files = sorted(globlib.glob(str(res_dir / "train" / "*.parquet")))
        print(f"  Warning: no {split} split for {resolution}x{resolution}, using train")

    samples = []
    for f in files:
        table = pq.read_table(f)
        for i in range(min(n - len(samples), len(table))):
            row = table.slice(i, 1).to_pydict()
            # Decode the tensor
            tensor_data = decode_tensor_blob(row["tensor"][0])
            # tensor_data shape: (3, H, W) -> [elevation, markers, obstacles]
            elevation = tensor_data[0]  # raw elevation
            markers = tensor_data[1]
            obstacles = tensor_data[2] if tensor_data.shape[0] > 2 else np.zeros_like(elevation)

            # Extract start/end from markers
            start_pos = np.argwhere(np.isclose(markers, 0.0))
            end_pos = np.argwhere(markers > 0.5)

            if len(start_pos) < 1 or len(end_pos) < 1:
                continue

            start = (int(start_pos[0, 0]), int(start_pos[0, 1]))
            end = (int(end_pos[0, 0]), int(end_pos[0, 1]))

            samples.append(
                {
                    "elevation": elevation,
                    "start": start,
                    "end": end,
                    "obstacles": obstacles,
                }
            )

            if len(samples) >= n:
                break
        if len(samples) >= n:
            break

    print(f"  Loaded {len(samples)} samples at {resolution}x{resolution}")
    return samples


# Load samples for each resolution
test_samples = {}
for res in RESOLUTIONS:
    test_samples[res] = load_samples(res, split="test", n=N_SAMPLES)

## 2. Run Timing Benchmark

For each (resolution, solver) pair, measure solve time over all samples.

In [ ]:
# Numba JIT warmup: run one solve to trigger compilation
if NUMBA_AVAILABLE and test_samples:
    warmup_res = RESOLUTIONS[0]
    warmup_sample = test_samples[warmup_res][0]
    print("Warming up Numba JIT...")
    _ = find_path(
        crop=warmup_sample["elevation"],
        start=warmup_sample["start"],
        end=warmup_sample["end"],
        solver_config=SOLVERS[0],
    )
    print("Warmup complete.")

In [ ]:
# Main benchmark loop
all_timings = []

for res in RESOLUTIONS:
    samples = test_samples.get(res, [])
    if not samples:
        print(f"No samples for {res}x{res}, skipping.")
        continue

    print(f"\n=== {res}x{res} ({len(samples)} samples) ===")

    for solver in SOLVERS:
        solver_name = solver["name"]
        times_ms = []
        successes = 0

        for i, sample in enumerate(samples):
            t0 = time.perf_counter()
            path = find_path(
                crop=sample["elevation"],
                start=sample["start"],
                end=sample["end"],
                solver_config=solver,
            )
            elapsed_ms = (time.perf_counter() - t0) * 1000
            times_ms.append(elapsed_ms)
            if len(path) > 0:
                successes += 1

            if (i + 1) % 25 == 0:
                print(f"  {solver_name}: {i + 1}/{len(samples)} done, mean so far = {np.mean(times_ms):.2f} ms")

        times_arr = np.array(times_ms)
        record = {
            "resolution": f"{res}x{res}",
            "solver": solver_name,
            "n_samples": len(samples),
            "success_rate": successes / len(samples),
            "mean_ms": np.mean(times_arr),
            "std_ms": np.std(times_arr),
            "median_ms": np.median(times_arr),
            "p95_ms": np.percentile(times_arr, 95),
            "p99_ms": np.percentile(times_arr, 99),
            "min_ms": np.min(times_arr),
            "max_ms": np.max(times_arr),
        }
        all_timings.append(record)
        print(
            f"  {solver_name}: mean={record['mean_ms']:.2f} ms, "
            f"median={record['median_ms']:.2f} ms, "
            f"p95={record['p95_ms']:.2f} ms, "
            f"success={record['success_rate']:.1%}"
        )

timings_df = pd.DataFrame(all_timings)
print("\n=== Full Results ===")
print(timings_df.to_string(index=False))

## 3. Visualization

In [ ]:
if not timings_df.empty:
    # Bar chart: mean solve time per solver per resolution
    fig, axes = plt.subplots(1, len(RESOLUTIONS), figsize=(7 * len(RESOLUTIONS), 5))
    if len(RESOLUTIONS) == 1:
        axes = [axes]

    fig.suptitle("A* Solver Inference Time by Resolution", fontsize=14, fontweight="bold")

    for ax, res in zip(axes, RESOLUTIONS):
        res_df = timings_df[timings_df["resolution"] == f"{res}x{res}"]
        if res_df.empty:
            continue
        bars = ax.bar(
            res_df["solver"], res_df["mean_ms"], yerr=res_df["std_ms"], capsize=5, color="steelblue", alpha=0.8
        )
        ax.set_ylabel("Time (ms)")
        ax.set_title(f"{res}x{res}")
        ax.grid(True, alpha=0.3, axis="y")

        # Add value labels
        for bar, val in zip(bars, res_df["mean_ms"]):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.1f}", ha="center", va="bottom", fontsize=9
            )

    plt.tight_layout()
    plt.savefig(ROOT / "outputs" / "paper" / "astar_timing_bars.pdf", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No timing results to plot.")

In [ ]:
if not timings_df.empty:
    # Resolution scaling plot (log-log)
    fig, ax = plt.subplots(figsize=(10, 6))

    for solver_name in timings_df["solver"].unique():
        solver_df = timings_df[timings_df["solver"] == solver_name].copy()
        solver_df["res_int"] = solver_df["resolution"].str.split("x").str[0].astype(int)
        solver_df = solver_df.sort_values("res_int")

        ax.plot(solver_df["res_int"], solver_df["mean_ms"], "o-", label=solver_name, linewidth=2)
        ax.fill_between(
            solver_df["res_int"],
            solver_df["mean_ms"] - solver_df["std_ms"],
            solver_df["mean_ms"] + solver_df["std_ms"],
            alpha=0.15,
        )

    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("Resolution (pixels per side)")
    ax.set_ylabel("Solve Time (ms)")
    ax.set_title("A* Solver Scaling with Resolution")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Add reference lines for neural model times (placeholder)
    # ax.axhline(y=5, color="green", linestyle="--", alpha=0.5, label="~Neural model (5ms)")

    plt.tight_layout()
    plt.savefig(ROOT / "outputs" / "paper" / "astar_scaling.pdf", dpi=150, bbox_inches="tight")
    plt.show()

    # TODO:RESOLUTION_UPDATE: This plot becomes most compelling with all 7 resolutions

In [ ]:
if not timings_df.empty:
    # Distribution of solve times (histogram)
    fig, axes = plt.subplots(1, len(RESOLUTIONS), figsize=(7 * len(RESOLUTIONS), 4))
    if len(RESOLUTIONS) == 1:
        axes = [axes]

    fig.suptitle("Distribution of A* Solve Times (grid:astar)", fontsize=14, fontweight="bold")

    for ax, res in zip(axes, RESOLUTIONS):
        # Use grid:astar times (the reference solver)
        res_grid = timings_df[(timings_df["resolution"] == f"{res}x{res}") & (timings_df["solver"] == "grid:astar")]
        if res_grid.empty:
            continue

        mean_val = res_grid["mean_ms"].values[0]
        p95_val = res_grid["p95_ms"].values[0]

        ax.set_title(f"{res}x{res} (mean={mean_val:.1f}ms, p95={p95_val:.1f}ms)")
        ax.set_xlabel("Solve Time (ms)")
        ax.set_ylabel("Count")
        ax.axvline(x=mean_val, color="red", linestyle="--", label=f"Mean: {mean_val:.1f}ms")
        ax.axvline(x=p95_val, color="orange", linestyle=":", label=f"P95: {p95_val:.1f}ms")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 4. Export Results

Save timing results for use in the passage-vision experiment notebook.

In [ ]:
if not timings_df.empty:
    # Save as CSV
    output_dir = ROOT / "outputs" / "paper"
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / "astar_inference_timing.csv"
    timings_df.to_csv(csv_path, index=False)
    print(f"Saved timing results to {csv_path}")

    # Also save as JSON for easy loading in passage-vision notebook
    json_path = output_dir / "astar_inference_timing.json"
    timing_dict = {}
    for _, row in timings_df[timings_df["solver"] == "grid:astar"].iterrows():
        res_int = int(row["resolution"].split("x")[0])
        timing_dict[res_int] = {
            "mean": round(row["mean_ms"], 2),
            "std": round(row["std_ms"], 2),
            "median": round(row["median_ms"], 2),
            "p95": round(row["p95_ms"], 2),
            "p99": round(row["p99_ms"], 2),
            "n_samples": int(row["n_samples"]),
        }

    with open(json_path, "w") as f:
        json.dump(timing_dict, f, indent=2)
    print(f"Saved JSON timing for passage-vision: {json_path}")

    print("\nTo load in passage-vision notebook:")
    print(f'  import json; ASTAR_TIMING = json.load(open("{json_path}"))')
else:
    print("No results to export. Run the benchmark cells above first.")

### Analysis: A* Solver Timing

**Expected scaling behavior:**

- **64x64:** Grid A* should complete in ~5-20 ms. At this resolution, A* is cheap enough that neural surrogates offer only modest speedup.

- **256x256:** Grid A* should take ~50-500 ms depending on terrain complexity and path length. This is the resolution where neural surrogates start to become compelling.

- **512x512+:** Grid A* scales roughly as O(N^2 log N). At 1024x1024, expect ~1-10 seconds. At 4096x4096, expect minutes. This is where neural surrogates provide transformative speedup.

**Numba JIT impact:** The grid backend with Numba should be 10-100x faster than NetworkX at the same resolution. This is itself a contribution of the paper (Section 5).

**Key paper figure:** The scaling plot (log-log: resolution vs time) with grid and networkx curves diverging at higher resolutions. Overlay neural model inference times to show the crossover point.

<!-- TODO:RESOLUTION_UPDATE: The full 64->4096 scaling curve is the most impactful figure. -->